In [ ]:
# ===== 環境セットアップ =====
import os, subprocess

def _is_kaggle():
    return os.path.exists("/kaggle/input")

def _is_runpod():
    return os.environ.get("RUNPOD_POD_ID") is not None

ENV = "kaggle" if _is_kaggle() else "runpod" if _is_runpod() else "local"
print(f"[env] {ENV}")

if ENV == "runpod":
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    import json as _json
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as _f:
        _json.dump({"username": os.environ["KAGGLE_USERNAME"],
                    "key": os.environ["KAGGLE_KEY"]}, _f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)


In [ ]:
# ===== パス定義 =====
import glob as _glob

if ENV == "kaggle":
    MODEL_DIR        = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
    DATA_DIR         = "/kaggle/input/nvidia-nemotron-model-reasoning-challenge"
    OUTPUT_DIR       = "/kaggle/working/adapter"
elif ENV == "runpod":
    MODEL_DIR        = "/workspace/models/nemotron-3-nano-30b-a3b-bf16"
    DATA_DIR         = "/workspace/data/nvidia-nemotron-competition"
    OUTPUT_DIR       = "/workspace/output/adapter"
else:  # local
    MODEL_DIR        = None
    DATA_DIR         = "data/nvidia-nemotron-competition"
    OUTPUT_DIR       = "output/adapter"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"MODEL_DIR  = {MODEL_DIR}")
print(f"DATA_DIR   = {DATA_DIR}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")


In [ ]:
# ===== ライブラリインストール =====
# Kaggle には基本ライブラリが入っているが念のため確認
!pip install -q peft trl bitsandbytes accelerate


In [ ]:
# ===== データ確認 =====
import pandas as pd

train_csv = f"{DATA_DIR}/train.csv"
train_df = pd.read_csv(train_csv)
print(f"train shape: {train_df.shape}")
print(train_df.head(3))


In [ ]:
# ===== 学習実行 =====
# パラメータをここで調整する

LORA_RANK  = 16
EPOCHS     = 2
LR         = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8
MAX_SEQ    = 2048

!python train.py \
    --model_dir  "{MODEL_DIR}" \
    --data_dir   "{DATA_DIR}" \
    --output_dir "{OUTPUT_DIR}" \
    --lora_rank  {LORA_RANK} \
    --epochs     {EPOCHS} \
    --lr         {LR} \
    --batch_size {BATCH_SIZE} \
    --grad_accum {GRAD_ACCUM} \
    --max_seq_len {MAX_SEQ} \
    --use_4bit


In [ ]:
# ===== 出力確認 =====
import os
print(os.listdir(OUTPUT_DIR))
